# Uncertainty Quantification for LLMs: A Comprehensive Tutorial

Welcome to this interactive tutorial on estimating uncertainty in Large Language Models (LLMs) applied to the clinical domain.

This notebook provides a hands-on approach to Uncertainty Quantification (UQ) by leveraging two state-of-the-art Python libraries: **`lm-polygraph`** and **`uqlm`**.

##### Load  the requested libraries

In [1]:
# CELL 0: ENVIRONMENT SETUP (Run this first!)
!pip install -q lm-polygraph uqlm transformers accelerate langchain langchain-huggingface langchain_openai

In [ ]:
import lm_polygraph
import uqlm
import torch
import random
import numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer
import os
import getpass


##### Fix a random seed to ensure reproducibility of your experiment

In [ ]:
# --- Setting seed for Scientific Reproducibility ---
RANDOM_SEED = 42

def set_global_seed(seed=RANDOM_SEED):
    """Locks the random seed for predictable, reproducible results."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_global_seed()


## Model selection
### Setup the Control Panel
In this first phase, we lay the foundations for our *Uncertainty Quantification* experiment. To ensure maximum reproducibility and clean code, we have created a **centralized Control Panel**. By modifying the four main variables, you can reconfigure the entire architecture without touching the underlying logic.

**1. Imports (The Wrappers)**
* We import the `BlackboxModel` and `WhiteboxModel` classes for the `lm_polygraph` library.
* We import the *Scorers* for the `uqlm` library, alongside the **LangChain** adapters (`ChatOpenAI`, `ChatHuggingFace`). LangChain acts as a universal bridge, allowing `uqlm` to communicate with any external API using a standard format.

**2. Configuration Variables**
* `PROVIDER`: The main switch. Choose `"openai"` or `"huggingface"` to route all requests to the respective servers.
  > 💡 **Why only these two providers?** While `uqlm` (thanks to LangChain) supports dozens of API providers, `lm_polygraph` currently only supports OpenAI and Hugging Face for Black/Grey-box inference. We deliberately restricted the scope to these two providers to ensure a rigorously fair, 1:1 comparative analysis between the two libraries.
* `MODEL_ID`: The exact name of the model you want to test (e.g., `"google/gemma-2-2b-it"` or `"gpt-3.5-turbo"`).
* `POLYGRAPH_MODE`: Choose `"white"` to download the model's weights into local memory (VRAM), unlocking attention matrices and hidden states, or `"black"` to query the model via API.
* `UQLM_MODE`: Choose `"white"` to force the APIs to return token probabilities (*Logprobs*), or `"black"` to evaluate uncertainty purely by analyzing the generated text.

> ⚠️ **Crucial Note on UQ Availability and Access Levels:** > The mode you select directly dictates the arsenal of Uncertainty Quantification techniques at your disposal. This is a strict hierarchy:
> * **White-Box Mode (Total Access):** By downloading the weights locally, you gain complete mathematical access to the model's internals. Because you have the highest level of access, a White-box model can execute **ALL** UQ methods (White-box  and Black-box).
> * **Black-Box Mode (Restricted Access):** This restricts access purely to the external API outputs. Consequently, you are limited **ONLY** to Black-box statistical methods (like Self-Consistency or text-based semantic similarity). You cannot apply attention-based or entropy-based methods here unless the API explicitly supports probability extraction (Grey-box).

In [ ]:
# --- Imports for Wrapper A (lm_polygraph) ---
from lm_polygraph.utils.model import BlackboxModel, WhiteboxModel
# --- Imports for Wrapper B (uqlm via LangChain) ---
from uqlm.scorers.black_box import BlackBoxUQ
from uqlm.scorers.white_box import WhiteBoxUQ
from uqlm.scorers.longtext import LongTextUQ

from langchain_openai import ChatOpenAI
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint

PROVIDER = "huggingface"          # Options: "openai" or "huggingface"
MODEL_ID = "google/gemma-2-2b-it" # Target model (e.g., "gpt-3.5-turbo" or "google/gemma-2-2b-it")


In [ ]:
while True:
    POLYGRAPH_MODE = input("Select POLYGRAPH_MODE ('white' or 'black'): ").strip().lower()
    if POLYGRAPH_MODE in ['white', 'black']:
        break
    print("Invalid input. Please type exactly 'white' or 'black'.")

# 2. Ask for UQLM Mode with validation
while True:
    UQLM_MODE = input("Select UQLM_MODE ('white' or 'black'): ").strip().lower()
    if UQLM_MODE in ['white', 'black']:
        break
    print("Invalid input. Please type exactly 'white' or 'black'.")

### Sanity Checks & Secure Authentication
Before allocating heavy resources or initiating network requests, this block ensures that the chosen configuration is logically sound and securely authenticated. This proactive approach prevents unexpected runtime crashes.

**1. Architectural Sanity Checks**
Not all configurations are physically possible. For instance, OpenAI models (like GPT-4 or GPT-3.5) are proprietary and closed-source. It is impossible to download their weights into your local VRAM. If a user accidentally sets `PROVIDER = "openai"` alongside `POLYGRAPH_MODE = "white"`, the script will immediately catch the logical conflict and raise a clear `ValueError`, guiding the user to correct the setup.

**2. Dynamic and Secure Credential Injection**
Security and efficiency are paramount. Instead of hardcoding sensitive API keys or asking for tokens you don't need:
* The script evaluates your `PROVIDER` and `MODE` choices.
* It dynamically prompts you **only** for the specific credentials required by your active configuration (e.g., it won't ask for a Hugging Face token if you are only querying the OpenAI API).
* It uses the `getpass` module to securely mask your input, ensuring that your private keys are never exposed in the notebook's output history or saved state.

In [ ]:
print(f"Initializing Master Architecture for {MODEL_ID}...")

# Prevent impossible architectural states
if PROVIDER == "openai" and POLYGRAPH_MODE == "white":
    raise ValueError("Conflict: OpenAI models cannot be downloaded to local VRAM. Set POLYGRAPH_MODE='black'.")

# Request exactly the keys needed for the chosen configuration
if PROVIDER == "openai" and "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Paste your OpenAI API Key: ")

if (PROVIDER == "huggingface" or POLYGRAPH_MODE == "white") and "HF_TOKEN" not in os.environ:
    os.environ["HF_TOKEN"] = getpass.getpass("Paste your Hugging Face Token: ")

###Building the UQ Wrappers (Polygraph & UQLM)
Now we bring the architecture to life by constructing the specific **Wrappers** required by each library. These wrappers act as the crucial translation layer, converting raw LLM outputs into a format that the Uncertainty Quantification (UQ) algorithms can mathematically process.

**1. Wrapper A: The `lm_polygraph` Interfaces**
* **`WhiteboxModel`:** This wrapper directly ingests the raw, locally downloaded model weights (`base_model`) and its `tokenizer`. By wrapping the physical model, it exposes the deepest mathematical layers of the LLM—such as hidden states, attention matrices, and logits—directly to Polygraph's advanced estimators.
* **`BlackboxModel`:** Instead of local weights, this wrapper encapsulates an external API connection. While it typically treats the LLM as a pure text-in/text-out oracle, setting `supports_logprobs=True` gracefully upgrades it into a "Grey-box." This allows Polygraph to compute token-level entropy using API probabilities without ever needing the physical model.

**2. Wrapper B: The `uqlm` Interfaces (via LangChain)**
* **The LangChain Bridge:** Unlike Polygraph, `uqlm` does not interface with models directly. It requires a LangChain adapter (`ChatOpenAI` or `ChatHuggingFace`) to wrap the model, standardizing the connection regardless of the underlying provider.
* **`WhiteBoxUQ` Scorer:** In the `uqlm` dictionary, "White-box" means probability-based. This wrapper takes the LangChain object and leverages extracted `logprobs` from the API, enabling fast, single-generation mathematical scoring.
* **`BlackBoxUQ` Scorer:** This wrapper ignores probabilities entirely. It wraps the model to perform purely text-based sampling UQ, evaluating uncertainty based on the semantic consistency across multiple generated responses.

In [ ]:
from lm_polygraph.utils.generation_parameters import GenerationParameters

print(f" Setting up lm_polygraph in {POLYGRAPH_MODE.upper()}-BOX mode...")
shared_generation_params = GenerationParameters()
shared_generation_params.temperature = 0.7
shared_generation_params.do_sample = True
shared_generation_params.max_new_tokens = 256

if POLYGRAPH_MODE == "white":
    # Load heavy weights into GPU exactly once (16-bit precision)
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=os.environ["HF_TOKEN"])
    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        device_map="cuda:0",
        torch_dtype=torch.float16,
        token=os.environ["HF_TOKEN"]
    )
    polygraph_model = WhiteboxModel(base_model, tokenizer, model_path=MODEL_ID, generation_parameters=shared_generation_params)
else:
  try:
        polygraph_model = BlackboxModel(
            model_path=MODEL_ID,
            openai_api_key=os.environ.get("OPENAI_API_KEY"),
            hf_api_token=os.environ.get("HF_TOKEN"),
            supports_logprobs= True,
            generation_parameters=shared_generation_params
        )
  except Exception as e:
        # Catching the exception
        print(f"WARNING: Failed to initialize BlackboxModel with logprobs. Error: {e}")
        print("Attempting graceful fallback to pure Black-box mode (supports_logprobs=False)...")

        # Fallback initialization without logprobs
        polygraph_model = BlackboxModel(
            model_path=MODEL_ID,
            openai_api_key=os.environ.get("OPENAI_API_KEY"),
            hf_api_token=os.environ.get("HF_TOKEN"),
            supports_logprobs=False,
            generation_parameters=shared_generation_params
        )

In [ ]:
from transformers import pipeline
from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace
import types
import asyncio
from langchain_core.messages import SystemMessage, HumanMessage


print(f"📊 Setting up uqlm in {UQLM_MODE.upper()}-BOX mode...")

# Build the shared LangChain bridge
if PROVIDER == "openai":
    langchain_llm = ChatOpenAI(
        model=MODEL_ID,
        api_key=os.environ["OPENAI_API_KEY"],
        model_kwargs={"logprobs": True} if UQLM_MODE == "white" else {}
    )

elif PROVIDER == "huggingface":
    if UQLM_MODE == "white":
        print("   ↳ Bridging the LOCAL GPU model directly to LangChain...")
        pipe = pipeline(
            "text-generation",
            model=base_model,
            tokenizer=tokenizer,
            max_new_tokens=256,
            return_full_text=False,
            do_sample=True,
            temperature=0.7
        )

        local_hf_llm = HuggingFacePipeline(pipeline=pipe)

        langchain_llm = ChatHuggingFace(llm=local_hf_llm)

    else:
        print("   ↳ Using Remote HuggingFace API (Warning: May be unstable for async UQ)...")
        hf_endpoint = HuggingFaceEndpoint(
            repo_id=MODEL_ID,
            huggingfacehub_api_token=os.environ["HF_TOKEN"],
            task="text-generation",
            temperature=0.7,
            max_new_tokens=256,
            model_kwargs={
                "do_sample": True,
                "return_full_text": False
            }
        )
        langchain_llm = ChatHuggingFace(llm=hf_endpoint)



print("  Applying async patch")

gpu_lock = asyncio.Lock()

async def _agenerate_shim(self, messages, stop=None, run_manager=None, **kwargs):
    sanitized_messages = []
    system_buffer = ""

    for msg in messages:
        if isinstance(msg, SystemMessage):
            system_buffer += msg.content + "\n\n"
        elif isinstance(msg, HumanMessage):
            if system_buffer:
                msg.content = system_buffer + msg.content
                system_buffer = ""
            sanitized_messages.append(msg)
        else:
            sanitized_messages.append(msg)
    # --------------------------------------------------------------------

    async with gpu_lock:
        return await asyncio.to_thread(
            self._generate,
            sanitized_messages,
            stop=stop,
            run_manager=run_manager,
            **kwargs
        )

langchain_llm._agenerate = types.MethodType(_agenerate_shim, langchain_llm)



### UQEngineContext

A centralized **Context Object** that manages the configuration and dependencies for the UQ framework.

* **State & Models:** Stores the execution modes (`"white"` or `"black"`) and holds the initialized model instances for both Polygraph and UQLM.
* **Auto-Validation:** Uses `__post_init__` to automatically enforce valid modes upon instantiation, ensuring a strict *fail-fast* architecture.

In [ ]:
from dataclasses import dataclass
from typing import Any, Optional

@dataclass
class UQEngineContext:
    polygraph_mode: str  # "white" o "black"
    uqlm_mode: str       # "white" o "black"
    polygraph_model: Optional[Any] = None
    langchain_llm: Optional[Any] = None

    def __post_init__(self):
        if self.polygraph_mode not in ["white", "black"]:
            raise ValueError("'polygraph_mode' must be 'white' o 'black'.")
        if self.uqlm_mode not in ["white", "black"]:
            raise ValueError("'uqlm_mode' must be 'white' o 'black'.")

In [ ]:
uq_engine = UQEngineContext(
    polygraph_mode="white",
    uqlm_mode="white",
    polygraph_model=polygraph_model,
    langchain_llm=langchain_llm
)

### UQ_REGISTRY & Access Hierarchy

A centralized dictionary that configures and routes all supported Uncertainty Quantification (UQ) techniques across different libraries.

* **Hierarchical Access (`MODE_LEVELS`):** Assigns numerical privilege levels (`black: 0`, `white: 1`) to enforce strict authorization, ensuring models meet the minimum required access to run a specific technique.


In [ ]:
# --- Imports for lm_polygraph Execution ---
from lm_polygraph.estimators import *
from lm_polygraph import estimate_uncertainty

# ==========================================
# THE UQ REGISTRY (
# ==========================================

# Defining the hierarchy:
MODE_LEVELS = {
    "black": 0,
    "white": 1
}

UQ_REGISTRY = {
    # --- lm_polygraph techniques  ---
    "polygraph_attention": {
        "library": "lm_polygraph",
        "supported_granularity": ["sequence", "claim"],
        "min_required_mode": "white",
        "estimator_class": AttentionScore,
        "description": "Estimates uncertainty based on model’s attention weights."
    },
    "polygraph_max_token_prob": {
        "library": "lm_polygraph",
        "supported_granularity": ["token"],
        "min_required_mode": "black",
        "estimator_class": MaximumTokenProbability,
        "description": "Estimates token-level uncertainty by calculating log-probability."
    },

    # --- uqlm techniques ---
    "uqlm_exact_match": {
        "library": "uqlm",
        "supported_granularity": ["sequence"],
        "min_required_mode": "black",
        "wrapper_class": BlackBoxUQ,
        "description": "Black-box technique computing exact match consistency."
    },
    "uqlm_entailment": {
        "library": "uqlm",
        "supported_granularity": ["sequence", "claim"],
        "wrapper_class": {
            "sequence": BlackBoxUQ,
            "claim": LongTextUQ
        },
        "description": "Entailment Probability computes mean entailment via an NLI model."
    }
}

###  `show_help` (CLI Discovery Utility)

An internal helper function that enhances framework **discoverability** by rendering a clean, tabular summary of all registered UQ techniques directly in the terminal.

* **Dynamic Filtering:** Allows users to narrow down the available techniques by passing optional search parameters (`library`, `mode`, `granularity`).

In [ ]:
def show_help(library: str = None, mode: str = None, granularity: str = None):
    """
    Internal CLI/Help function.
    Prints a formatted summary table of all registered UQ techniques.

    Optional parameters to filter the output:
    - library (str): e.g., "uqlm" or "lm_polygraph"
    - mode (str): e.g., "white" or "black" (filters by minimum requirement)
    - granularity (str): e.g., "token", "sequence", "claim"
    """
    print("\n🔍 UQ FRAMEWORK - AVAILABLE TECHNIQUES")

    # --- 1. Table Structure Definition ---
    # We use f-string alignment modifiers (e.g., <25 means "left-align, occupy 25 characters")
    header = f"{'TECHNIQUE':<26} | {'LIBRARY':<13} | {'MODE':<6} | {'GRANULARITY':<18} | {'DESCRIPTION'}"
    separator = "-" * 120

    print(separator)
    print(header)
    print(separator)

    count = 0
    for tech_name, info in UQ_REGISTRY.items():
        # --- 2. Safe Extraction (Fail-Safe) ---
        lib = info.get("library", "N/A")
        req_mode = info.get("min_required_mode", "N/A")
        gran = ", ".join(info.get("supported_granularity", []))
        desc = info.get("description", "No description provided.")

        # --- 3. Filtering Engine ---
        if library and lib.lower() != library.lower():
            continue

        if mode and req_mode.lower() != mode.lower():
            continue

        if granularity and granularity.lower() not in [g.lower() for g in info.get("supported_granularity", [])]:
            continue

        # --- 4. Visual Cleanup ---
        # Truncate the description if it's too long to keep the table clean in the terminal
        max_desc_len = 200
        if len(desc) > max_desc_len:
            desc = desc[:max_desc_len - 3] + "..."

        # Print the formatted row in columns
        row = f"{tech_name:<26} | {lib:<13} | {req_mode:<6} | {gran:<18} | {desc}"
        print(row)
        count += 1

    print(separator)
    print(f"Showing {count} techniques based on applied filters.\n")

In [ ]:
show_help()

### The Core Execution Engine (Dispatcher & Handlers)

A state-of-the-art routing architecture that separates validation logic from library-specific execution using the **Facade + Handlers** pattern.

* **`evaluate_uncertainty` (The Dispatcher):** The universal public interface to **compute the uncertainty score**. It performs *fail-fast* registry validation, enforces hierarchical access controls (preventing Privilege Escalation between `white` and `black` modes), and securely routes the request to the correct underlying library to generate the standardized UQ payload.
* **`_handle_polygraph_execution`:** The private sub-engine for `lm_polygraph`. It handles its specific synchronous API, complex token string decoding, and multi-step claim extraction pipelines.
* **`_handle_uqlm_execution`:** The private asynchronous sub-engine for `uqlm`. It dynamically resolves the correct wrapper class based on the requested granularity and parses the library's nested output dictionaries.

In [ ]:
# ==========================================
#  AUXILIARY FUNCTIONS (Private logic)
# ==========================================
import os
import getpass
from lm_polygraph.stat_calculators import GreedyProbsCalculator, ClaimsExtractor
from lm_polygraph.utils.openai_chat import OpenAIChat

def _evaluate_claim_level_polygraph(prompt: str, estimator_class, model):
    """
    Handles the complex multi-step pipeline for Claim-level UQ in lm_polygraph.
    Requires OPENAI_API_KEY in the environment for the ClaimsExtractor.
    """
    print("   ↳ Initiating multi-step Claim-Level Pipeline...")

    if "OPENAI_API_KEY" not in os.environ or not os.environ["OPENAI_API_KEY"]:
        print("\n   ⚠️ Warning: Missing Open AI Key for ClaimsExtractor.")
        api_key = getpass.getpass("    Insert Open-AI key (sk-...): ")
        os.environ["OPENAI_API_KEY"] = api_key.strip()
        print("   Open AI key set!\n")
    # ---------------------------------------------

    stat = {}
    texts = [prompt]

    print("   ↳ Step 1: Generating text and probabilities (GreedyProbsCalculator)...")
    greedy_calc = GreedyProbsCalculator()
    stat.update(greedy_calc(stat, texts, model))

    print("   ↳ Step 2: Extracting atomic claims (ClaimsExtractor via GPT-4)...")
    extractor = ClaimsExtractor(OpenAIChat("gpt-4"))
    stat.update(extractor(stat, texts, model))

    print(f"   ↳ Step 3: Applying {estimator_class.__name__}...")
    estimator = estimator_class()
    uncertainties = estimator(stat)


    print("   ↳ Step 4: Formatting the output...")
    claims_list = stat["claims"][0]
    scores_list = uncertainties[0]

    claim_details = []
    for claim_obj, score in zip(claims_list, scores_list):
        claim_details.append({
            "claim_text": claim_obj.claim_text,
            "score": float(score)
        })

    result_payload = {
        "input_prompt": prompt,
        "generated_text": stat["greedy_texts"][0],
        "uncertainty_score": claim_details
    }

    return result_payload

# ==========================================
# 2. PRIVATE HANDLERS (The Sub-Engines)
# ==========================================

def _handle_polygraph_execution(prompt: str, tech_info: dict, granularity: str, polygraph_model, **kwargs):
    """Handles all execution and parsing specifically for lm_polygraph."""
    print(f"⏳ Routing to Wrapper A (lm_polygraph) -> {tech_info['estimator_class'].__name__}...")

    # --- CLAIM ---
    if granularity == "claim":
        result_payload = _evaluate_claim_level_polygraph(prompt, tech_info["estimator_class"](**kwargs), polygraph_model)
        result_payload["library"] = "lm_polygraph"
        result_payload["estimator_name"] = tech_info["estimator_class"].__name__
        result_payload["granularity"] = granularity
        return result_payload

    # --- TOKEN ---
    elif granularity == "token":
        estimator = tech_info["estimator_class"](**kwargs)
        output = estimate_uncertainty(polygraph_model, estimator, input_text=prompt)
        import numpy as np

        raw_score = output.uncertainty
        if isinstance(raw_score, list) and len(raw_score) > 0 and isinstance(raw_score[0], np.ndarray):
            clean_score_list = raw_score[0].tolist()
        elif isinstance(raw_score, np.ndarray):
            clean_score_list = raw_score.tolist()
        else:
            clean_score_list = list(raw_score)

        raw_tokens = output.generation_tokens
        if len(raw_tokens) == 1 and isinstance(raw_tokens[0], list):
            raw_tokens = raw_tokens[0]

        if len(raw_tokens) > 0 and isinstance(raw_tokens[0], int):
            token_strings = polygraph_model.tokenizer.convert_ids_to_tokens(raw_tokens)
        else:
            token_strings = raw_tokens

        token_details = []
        min_len = min(len(token_strings), len(clean_score_list))
        for i in range(min_len):
            clean_token = str(token_strings[i]).replace("Ġ", " ").replace(" ", " ")
            token_details.append({
                "token": clean_token,
                "score": float(clean_score_list[i])
            })

        return {
            "library": "lm_polygraph",
            "estimator_name": output.estimator,
            "granularity": granularity,
            "input_prompt": output.input_text,
            "generated_text": output.generation_text,
            "uncertainty_score": token_details,
        }

    # --- SEQUENCE ---
    else:
        estimator = tech_info["estimator_class"](**kwargs)
        output = estimate_uncertainty(polygraph_model, estimator, input_text=prompt)
        import numpy as np

        if isinstance(output.uncertainty, np.ndarray) or isinstance(output.uncertainty, list):
            final_score = float(output.uncertainty[0])
        else:
            final_score = float(output.uncertainty)

        return {
            "library": "lm_polygraph",
            "estimator_name": output.estimator,
            "granularity": granularity,
            "input_prompt": output.input_text,
            "generated_text": output.generation_text,
            "uncertainty_score": final_score
        }

In [ ]:
async def _handle_uqlm_execution(prompt: str, technique_name: str, tech_info: dict, granularity: str, langchain_llm, **kwargs):
    """Handles all execution and parsing specifically for uqlm."""

    wrapper_map = tech_info["wrapper_class"]
    print(f"⏳ Routing to Wrapper B (uqlm) -> {uqlm_class.__name__} with scorer: '{technique_name}'...")

    uqlm_class = wrapper_map[granularity] if isinstance(wrapper_map, dict) else wrapper_map

    uqlm_wrapper = uqlm_class(
        llm=langchain_llm,
        scorers=[technique_name],
        **kwargs
    )

    uqlm_result = await uqlm_wrapper.generate_and_score(prompts=[prompt])
    res_dict = uqlm_result.to_dict()

    # --- CLAIM ---
    if granularity == "claim":
        print(f"res dict: {res_dict}")
        raw_claims_data = res_dict["data"]["claims_data"][0]
        claim_details = []
        for c in raw_claims_data:
            claim_details.append({
                "claim_text": c['claim'],
                "score": c[technique_name]
            })

        return {
            "library": "uqlm",
            "estimator_name": technique_name,
            "granularity": granularity,
            "input_prompt": prompt,
            "generated_text": res_dict["data"]["responses"],
            "uncertainty_score": claim_details
        }

    # --- TOKEN ---
    elif granularity == "token":
        return "UQLM do not support token level granularuty"

    # --- SEQUENCE ---
    else:
        return {
            "library": "uqlm",
            "estimator_name": technique_name,
            "granularity": granularity,
            "input_prompt": prompt,
            "generated_text": res_dict["data"]["responses"],
            "uncertainty_score": res_dict["data"][technique_name][0]
        }



In [ ]:
async def evaluate_uncertainty(prompt: str, technique_name: str, library: str, granularity: str,
                         uq_context: UQEngineContext, **kwargs):
    """
    Universal interface for UQ evaluation. Routes to simple functions or
    complex pipelines based on the requested granularity, standardizing the output.
    """
    print(f"\n🧠 Processing Request: Library='{library}' | Technique='{technique_name}' | Granularity='{granularity}'")

    registry_key = f"{library}_{technique_name}"

    # --- Step A: Registry Validation ---
    if registry_key not in UQ_REGISTRY:
        raise ValueError(f"Technique combination '{registry_key}' is not in the UQ_REGISTRY. Please add it first.")

    tech_info = UQ_REGISTRY[registry_key]

    if granularity not in tech_info["supported_granularity"]:
        raise ValueError(
            f" Granularity Mismatch: '{technique_name}' in {library} only supports {tech_info['supported_granularity']}. "
            f"You requested '{granularity}'."
        )

    current_mode = getattr(uq_context, f"{library}_mode")
    min_required_mode = tech_info["min_required_mode"]

    # Translate mode strings into their corresponding numeric levels
    current_level = MODE_LEVELS.get(current_mode, -1)
    required_level = MODE_LEVELS.get(min_required_mode, 99)

    # If the current level is lower than the required one, block execution!
    if current_level < required_level:
        raise ValueError(
            f"Privilege Escalation Error: The technique '{registry_key}' requires "
            f"'{min_required_mode.upper()}' level access, but the library '{library}' "
            f"is initialized at a lower level ('{current_mode.upper()}')."
        )
    # ----------------------------------------------

    print(f"Validation Passed. Library to use: {tech_info['library'].upper()} (Mode: {current_mode})")

    # --- Step B: Execution Routing ---
    if tech_info["library"] == "lm_polygraph":
        if uq_context.polygraph_model is None:
            raise ValueError("'polygraph_model' is required by this technique but was not found in the UQEngineContext.")

        result_payload = _handle_polygraph_execution(prompt, tech_info, granularity, uq_context.polygraph_model, **kwargs)

    elif tech_info["library"] == "uqlm":
        if uq_context.langchain_llm is None:
              raise ValueError(" 'langchain_llm' bridge is required by this technique but was not found in the UQEngineContext.")

        result_payload = await _handle_uqlm_execution(prompt, technique_name, tech_info, granularity, uq_context.langchain_llm, **kwargs)

    print(f"🎯 {granularity.capitalize()}-level execution complete!")
    return result_payload

In [ ]:
test_result = await evaluate_uncertainty(
    prompt="What are the early signs of Parkinson's disease?",
    library="uqlm",
    technique_name="entailment",
    granularity="claim",
    uq_context=uq_engine
)
print(test_result)


# Text-Only

## Black Box Techniques

### Black-Box Section Setup

Run the cells below **before** the demonstrations. They are specific to this notebook and are not part of the shared `uq_tutorial.ipynb` infrastructure.

| Cell | Purpose |
|---|---|
| `bb-setup-pip` | Installs `scipy` (needed for Spearman correlation) |
| `bb-setup-imports` | Explicitly imports the black-box estimator classes |
| `bb-setup-engine` | Re-creates `uq_engine` using the modes you selected above (`POLYGRAPH_MODE` / `UQLM_MODE`), overriding the hardcoded `"white"` engine from the shared section |
| `bb-uqlm-handler` | Overrides shared `_handle_uqlm_execution` with a bug-fixed version: fixes `uqlm_class` NameError, splits `num_responses` vs scorer kwargs, converts confidence → uncertainty |
| `bb-evaluate-fn` | Overrides shared `evaluate_uncertainty` to merge `default_kwargs` from the registry |

> **Tip:** Select **`black`** for both `POLYGRAPH_MODE` and `UQLM_MODE` when prompted above to run all demos without local model weights.

In [ ]:
!pip install -q matplotlib scipy


In [ ]:
# ── Black-box-specific estimator imports ──────────────────────
from lm_polygraph.estimators import (
    # Verbalized
    Verbalized1S, Verbalized2S, Linguistic1S,
    # P(True) — verbalized self-evaluation
    PTrue, PTrueClaim,
    # Consistency — graph-free
    LexicalSimilarity, SemanticEntropy,
    NumSemSets, LabelProb,
    # Consistency — graph-based
    DegMat, Eccentricity, EigValLaplacian,
    # Consistency — SAR (Sentence-level Answer Relevance)
    SAR, SentenceSAR,
    # Consistency — Semantic Density
    SemanticDensity,
    # Claim-level
    FrequencyScoringClaim,
)
from scipy.stats import spearmanr
print("Black-box estimator classes loaded.")

In [ ]:
uq_engine = UQEngineContext(
    polygraph_mode=POLYGRAPH_MODE,
    uqlm_mode=UQLM_MODE,
    polygraph_model=polygraph_model,
    langchain_llm=langchain_llm,
)
print(f"UQ engine: polygraph={POLYGRAPH_MODE}, uqlm={UQLM_MODE}")


In [ ]:
async def _handle_uqlm_execution(prompt: str, technique_name: str, tech_info: dict, granularity: str, langchain_llm, **kwargs):
    """Handles all execution and parsing specifically for uqlm."""
    wrapper_map = tech_info["wrapper_class"]
    uqlm_class = wrapper_map[granularity] if isinstance(wrapper_map, dict) else wrapper_map
    print(f"⏳ Routing to Wrapper B (uqlm) -> {uqlm_class.__name__} with scorer: '{technique_name}'...")

    scorer_kwargs = {k: v for k, v in kwargs.items() if k not in ("num_responses", "sampling_temperature")}
    uqlm_wrapper = uqlm_class(llm=langchain_llm, scorers=[technique_name], **scorer_kwargs)

    gen_kwargs = {k: v for k, v in kwargs.items() if k in ("num_responses", "sampling_temperature")}
    uqlm_result = await uqlm_wrapper.generate_and_score(prompts=[prompt], **gen_kwargs)
    res_dict = uqlm_result.to_dict()

    if granularity == "claim":
        raw_claims_data = res_dict["data"]["claims_data"][0]
        claim_details = [{"claim_text": c["claim"], "score": 1.0 - float(c[technique_name])} for c in raw_claims_data]
        return {
            "library": "uqlm",
            "estimator_name": technique_name,
            "granularity": granularity,
            "input_prompt": prompt,
            "generated_text": res_dict["data"]["responses"],
            "uncertainty_score": claim_details,
        }
    elif granularity == "token":
        raise ValueError("UQLM does not support token-level granularity.")
    else:
        confidence = res_dict["data"][technique_name][0]
        return {
            "library": "uqlm",
            "estimator_name": technique_name,
            "granularity": granularity,
            "input_prompt": prompt,
            "generated_text": res_dict["data"]["responses"],
            "uncertainty_score": 1.0 - float(confidence),
            "confidence_score": float(confidence),
        }


In [ ]:
async def evaluate_uncertainty(prompt: str, technique_name: str, library: str, granularity: str,
                         uq_context: UQEngineContext, **kwargs):
    """Universal interface for UQ evaluation."""
    print(f"\n🧠 Processing Request: Library='{library}' | Technique='{technique_name}' | Granularity='{granularity}'")

    registry_key = f"{library}_{technique_name}"
    if registry_key not in UQ_REGISTRY:
        raise ValueError(f"Technique combination '{registry_key}' is not in the UQ_REGISTRY. Please add it first.")

    tech_info = UQ_REGISTRY[registry_key]
    if granularity not in tech_info["supported_granularity"]:
        raise ValueError(
            f"Granularity Mismatch: '{technique_name}' in {library} only supports "
            f"{tech_info['supported_granularity']}. You requested '{granularity}'."
        )

    current_mode = getattr(uq_context, f"{library}_mode")
    min_required_mode = tech_info.get("min_required_mode", "black")
    if MODE_LEVELS.get(current_mode, -1) < MODE_LEVELS.get(min_required_mode, 99):
        raise ValueError(
            f"Privilege Escalation Error: '{registry_key}' requires '{min_required_mode.upper()}' access, "
            f"but '{library}' is in '{current_mode.upper()}' mode."
        )

    print(f"Validation Passed. Library to use: {tech_info['library'].upper()} (Mode: {current_mode})")
    merged_kwargs = {**tech_info.get("default_kwargs", {}), **kwargs}

    if tech_info["library"] == "lm_polygraph":
        if uq_context.polygraph_model is None:
            raise ValueError("'polygraph_model' is required but missing from UQEngineContext.")
        result_payload = _handle_polygraph_execution(
            prompt, tech_info, granularity, uq_context.polygraph_model, **merged_kwargs
        )
    elif tech_info["library"] == "uqlm":
        if uq_context.langchain_llm is None:
            raise ValueError("'langchain_llm' is required but missing from UQEngineContext.")
        result_payload = await _handle_uqlm_execution(
            prompt, technique_name, tech_info, granularity, uq_context.langchain_llm, **merged_kwargs
        )

    print(f"🎯 {granularity.capitalize()}-level execution complete!")
    return result_payload


---
# Black-Box Uncertainty Quantification

Black-box UQ treats the LLM as an **opaque text API** — no logits, no hidden states, no attention weights.
This is the default deployment setting when querying GPT-4, Gemini, or any closed commercial model.

## Why It Matters in Clinical AI

In hospital decision-support systems, you typically integrate a third-party LLM via API.
Model internals are inaccessible. Black-box UQ is therefore the **most practically relevant** approach for clinical deployment: you can always estimate uncertainty regardless of whether you own or can access the model weights.

## Taxonomy (AAAI-2026 / TACL)

| Family | Core idea | Inference cost | Calibration on small LLMs |
|---|---|---|---|
| **Verbalized** | Ask the model *"how confident are you?"* | 1–2 calls | ⚠️ Unreliable below ~70B params |
| **Consistency** | Sample K answers; measure semantic agreement | K calls | ✅ Model-agnostic — works at any scale |

## Method Selection: Decision Tree

From the AAAI-2026 tutorial visual guide:

```
Black-box model?
  YES
  ├── Have compute budget?
  │     YES → Consistency-based: EigLaplacian, DegMat, Eccentricity, SemanticEntropy
  │     NO  → Verbalized uncertainty
```

**EigLaplacian is the recommended consistency method** from the decision tree — it is the primary technique covered in Section 2.3 below.

## Granularity in Black-Box Mode

| Granularity | Black-box? | Notes |
|---|---|---|
| **Sequence** | ✅ Both libraries | One score per full response |
| **Claim** | ✅ UQLM (`LongTextUQ`) | Per-atomic-claim scores |
| **Token** | ❌ Not supported | Requires logits (white-box only) |

> 💡 **Convention:** UQLM returns **confidence** ∈ [0, 1]; we convert to **uncertainty** as `1 − confidence`
> so that all visualizations share the same direction: **higher = more uncertain**.

### Clinical Prompts

Reused across all black-box demonstrations.

In [ ]:
CLINICAL_PROMPTS = [
    "What is the normal resting heart rate for a healthy adult?",
    "What is the first-line treatment for type 2 diabetes?",
    "What are the early signs of Parkinson's disease?",
    "What is the recommended dose of aspirin for cardiovascular prevention?",
    "What is the most effective treatment for long COVID neurological symptoms?",
    "Describe the mechanism of action of tirzepatide on GIP and GLP-1 receptors.",
]
DEMO_PROMPT = CLINICAL_PROMPTS[2]
FACTUAL_PROMPT = CLINICAL_PROMPTS[0]
UNCERTAIN_PROMPT = CLINICAL_PROMPTS[4]
short_prompts = [p[:42] + "..." for p in CLINICAL_PROMPTS]
print(f"Demo prompt: {DEMO_PROMPT}")

### Visualization Utilities

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

CMAP_UNCERTAINTY = plt.cm.RdYlGn_r

def plot_sequence_comparison(results, labels, title="Sequence-Level Uncertainty"):
    arr = np.array(results, dtype=float)
    norm = (arr - arr.min()) / (arr.max() - arr.min() + 1e-9)
    colors = [CMAP_UNCERTAINTY(v) for v in norm]
    fig, ax = plt.subplots(figsize=(12, 4))
    bars = ax.bar(range(len(labels)), arr, color=colors, edgecolor="white", width=0.6)
    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=25, ha="right", fontsize=8)
    ax.set_ylabel("Uncertainty (↑ = less confident)")
    ax.set_title(title, fontsize=12, fontweight="bold")
    for bar, val in zip(bars, arr):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002, f"{val:.3f}", ha="center", va="bottom", fontsize=8)
    plt.tight_layout(); plt.show()

def plot_claim_uncertainty(result, title=""):
    scores = result["uncertainty_score"]
    claims = [s["claim_text"][:55] + ("..." if len(s["claim_text"]) > 55 else "") for s in scores]
    values = [s["score"] for s in scores]
    arr = np.array(values)
    norm = (arr - arr.min()) / (arr.max() - arr.min() + 1e-9)
    colors = [CMAP_UNCERTAINTY(v) for v in norm]
    fig, ax = plt.subplots(figsize=(14, max(4, len(claims) * 0.45)))
    ax.barh(range(len(claims)), arr, color=colors, edgecolor="white", height=0.7)
    ax.set_yticks(range(len(claims)))
    ax.set_yticklabels(claims, fontsize=8)
    ax.invert_yaxis()
    ax.set_xlabel("Uncertainty")
    ax.set_title(title or "Claim-Level Uncertainty", fontweight="bold")
    plt.tight_layout(); plt.show()

print("Visualization utilities loaded.")

### Extending `UQ_REGISTRY` with Black-Box Techniques

Registry key pattern: `f"{library}_{technique_name}"` → e.g. `evaluate_uncertainty(..., library="polygraph", technique_name="verbalized_1s")` maps to `"polygraph_verbalized_1s"`.

In [ ]:
BB_REGISTRY_ENTRIES = {
    # ── VERBALIZED (lm-polygraph) ────────────────────────────────────
    "polygraph_verbalized_1s": {
        "library": "lm_polygraph",
        "supported_granularity": ["sequence"],
        "min_required_mode": "black",
        "estimator_class": Verbalized1S,
        "default_kwargs": {"confidence_regex": r"Probability:\s*(\d+\.?\d*)", "name_postfix": "_top1"},
        "description": "Verbalized 1S: answer + numeric confidence in one generation (Tian et al., 2023)."
    },
    "polygraph_verbalized_2s": {
        "library": "lm_polygraph",
        "supported_granularity": ["sequence"],
        "min_required_mode": "black",
        "estimator_class": Verbalized2S,
        "default_kwargs": {"confidence_regex": r"Probability:\s*(\d+\.?\d*)", "name_postfix": "_top1"},
        "description": "Verbalized 2S: separate generation then confidence-estimation turns."
    },
    "polygraph_linguistic_1s": {
        "library": "lm_polygraph",
        "supported_granularity": ["sequence"],
        "min_required_mode": "black",
        "estimator_class": Linguistic1S,
        "description": "Linguistic 1S: maps hedging phrases ('I think', 'maybe') to a confidence score."
    },
    # ── CONSISTENCY — graph-free (lm-polygraph) ──────────────────────
    "polygraph_lexical_similarity": {
        "library": "lm_polygraph",
        "supported_granularity": ["sequence"],
        "min_required_mode": "black",
        "estimator_class": LexicalSimilarity,
        "default_kwargs": {"metric": "rougeL"},
        "description": "Mean ROUGE-L overlap across K sampled answers. Cheap but surface-level."
    },
    "polygraph_semantic_entropy": {
        "library": "lm_polygraph",
        "supported_granularity": ["sequence"],
        "min_required_mode": "black",
        "estimator_class": SemanticEntropy,
        "default_kwargs": {"class_probability_estimation": "frequency"},
        "description": "Cluster K answers by NLI equivalence, compute Shannon entropy over clusters (Kuhn et al., 2023)."
    },
    "polygraph_num_sem_sets": {
        "library": "lm_polygraph",
        "supported_granularity": ["sequence"],
        "min_required_mode": "black",
        "estimator_class": NumSemSets,
        "description": "Number of distinct semantic equivalence classes — simpler SE proxy."
    },
    "polygraph_label_prob": {
        "library": "lm_polygraph",
        "supported_granularity": ["sequence"],
        "min_required_mode": "black",
        "estimator_class": LabelProb,
        "description": "Frequency of the greedy answer among K samples — black-box MSP proxy."
    },
    # ── CONSISTENCY — graph-based (lm-polygraph) ─────────────────────
    "polygraph_degmat": {
        "library": "lm_polygraph",
        "supported_granularity": ["sequence"],
        "min_required_mode": "black",
        "estimator_class": DegMat,
        "default_kwargs": {"similarity_score": "NLI_score", "affinity": "entail"},
        "description": "Degree Matrix: mean node degree of the NLI semantic similarity graph (Lin et al., 2023)."
    },
    "polygraph_eccentricity": {
        "library": "lm_polygraph",
        "supported_granularity": ["sequence"],
        "min_required_mode": "black",
        "estimator_class": Eccentricity,
        "default_kwargs": {"similarity_score": "NLI_score", "affinity": "entail"},
        "description": "Eccentricity: longest shortest path in the semantic similarity graph (Lin et al., 2023)."
    },
    "polygraph_eig_val_laplacian": {
        "library": "lm_polygraph",
        "supported_granularity": ["sequence"],
        "min_required_mode": "black",
        "estimator_class": EigValLaplacian,
        "default_kwargs": {"similarity_score": "NLI_score", "affinity": "entail"},
        "description": "EigLaplacian: largest eigenvalue of graph Laplacian — recommended method from AAAI-2026 decision tree."
    },
    # ── CONSISTENCY (uqlm) ───────────────────────────────────────────
    "uqlm_semantic_negentropy": {
        "library": "uqlm",
        "supported_granularity": ["sequence"],
        "min_required_mode": "black",
        "wrapper_class": BlackBoxUQ,
        "description": "Semantic negentropy — normalized entropy over NLI-equivalence clusters."
    },
    "uqlm_cosine_sim": {
        "library": "uqlm",
        "supported_granularity": ["sequence"],
        "min_required_mode": "black",
        "wrapper_class": BlackBoxUQ,
        "description": "Mean embedding cosine similarity between original and sampled responses."
    },
    "uqlm_noncontradiction": {
        "library": "uqlm",
        "supported_granularity": ["sequence"],
        "min_required_mode": "black",
        "wrapper_class": BlackBoxUQ,
        "description": "Non-contradiction probability via NLI model across K samples."
    },
    # ── P(True) — verbalized self-evaluation (Kadavath et al., 2022) ──────
    "polygraph_p_true": {
        "library": "lm_polygraph",
        "supported_granularity": ["sequence"],
        "min_required_mode": "black",
        "estimator_class": PTrue,
        "description": "P(True): sample K answers then ask the model is this correct? — aggregates TRUE probability.",
    },
    "polygraph_p_true_claim": {
        "library": "lm_polygraph",
        "supported_granularity": ["claim"],
        "min_required_mode": "black",
        "estimator_class": PTrueClaim,
        "description": "P(True) at claim level: each extracted claim independently self-evaluated by the model.",
    },
    # ── SAR — Sentence-level Answer Relevance ────────────────────────
    "polygraph_sar": {
        "library": "lm_polygraph",
        "supported_granularity": ["sequence"],
        "min_required_mode": "black",
        "estimator_class": SAR,
        "description": "SAR: combined token+sentence relevance between question and K sampled answers (Kuhn et al., 2023).",
    },
    "polygraph_sentence_sar": {
        "library": "lm_polygraph",
        "supported_granularity": ["sequence"],
        "min_required_mode": "black",
        "estimator_class": SentenceSAR,
        "description": "SentenceSAR: sentence-level average relevance between question and sampled answers.",
    },
    # ── Semantic Density ────────────────────────────────────────────
    "polygraph_semantic_density": {
        "library": "lm_polygraph",
        "supported_granularity": ["sequence"],
        "min_required_mode": "black",
        "estimator_class": SemanticDensity,
        "description": "SemanticDensity: density of K sampled answers in embedding space — low density = high uncertainty.",
    },
    # ── Claim-level: Frequency Scoring ────────────────────────────
    "polygraph_frequency_scoring_claim": {
        "library": "lm_polygraph",
        "supported_granularity": ["claim"],
        "min_required_mode": "black",
        "estimator_class": FrequencyScoringClaim,
        "description": "FrequencyScoringClaim: fraction of K samples that NLI-confirm each extracted claim. Pure black-box.",
    },
}

UQ_REGISTRY.update(BB_REGISTRY_ENTRIES)
print(f"Registry now contains {len(UQ_REGISTRY)} black-box techniques.")
show_help(mode="black")

---
## 1. Verbalized Uncertainty

The model is **explicitly prompted** to express confidence — no sampling required.

### How Each Variant Works

| Method | Mechanism | lm-polygraph class |
|---|---|---|
| **Verbalized 1S** | Answer + confidence number in a single generation | `Verbalized1S` |
| **Verbalized 2S** | First generate the answer; then ask a *separate* follow-up question about confidence | `Verbalized2S` |
| **Linguistic 1S** | Parse hedging language ("I think", "possibly", "it is likely") and map phrases to a numeric confidence | `Linguistic1S` |

### Mathematical Formulation (Verbalized 1S / 2S)

The model is prompted to produce:
```
Guess: <answer>
Probability: <p>
```
The extracted probability $\hat{p} \in [0,1]$ is then converted to uncertainty:
$$U_{\text{verb}} = 1 - \hat{p}$$

### Why Verbalization Fails on Small LLMs

Small models (< 7B parameters) struggle because:
1. **Instruction-following gap** — they often ignore the exact `Probability: X.XX` format, producing free-form text the regex cannot parse.
2. **Miscalibration** — they tend to output `Probability: 1.0` (overconfident) regardless of actual uncertainty.
3. **Sycophancy** — they mimic the expected "confident expert" persona even when they should hedge.

> 💡 UQLM has no verbalized scorer. Its black-box core relies entirely on consistency — which is why UQLM typically outperforms lm-polygraph in pure black-box mode on small models.

In [ ]:
VERBALIZED_TEMPLATE = (
    "You are a clinical assistant. Answer the medical question concisely.\n"
    "Then state your confidence as a probability between 0.0 and 1.0.\n"
    "Format exactly:\nGuess: <your answer>\nProbability: <number>\n\n"
    "Question: {question}"
)

verb_prompt = VERBALIZED_TEMPLATE.format(question=DEMO_PROMPT)
print(verb_prompt)

In [ ]:
print("Running Verbalized 1S (lm-polygraph)...")
v1s = await evaluate_uncertainty(
    prompt=verb_prompt,
    library="polygraph", technique_name="verbalized_1s", granularity="sequence",
    uq_context=uq_engine
)
print(f"Uncertainty: {v1s['uncertainty_score']:.4f}")
print(f"Generated:\n{v1s['generated_text'][:400]}...")

In [ ]:
print("Running Verbalized 2S (lm-polygraph)...")
v2s = await evaluate_uncertainty(
    prompt=verb_prompt,
    library="polygraph", technique_name="verbalized_2s", granularity="sequence",
    uq_context=uq_engine
)
print(f"Uncertainty: {v2s['uncertainty_score']:.4f}")

In [ ]:
print("Running Linguistic 1S (lm-polygraph)...")
ling_prompt = f"As a medical expert, answer this question. You may use hedging if unsure.\n\nQuestion: {DEMO_PROMPT}"
ling = await evaluate_uncertainty(
    prompt=ling_prompt,
    library="polygraph", technique_name="linguistic_1s", granularity="sequence",
    uq_context=uq_engine
)
print(f"Uncertainty: {ling['uncertainty_score']:.4f}")

In [ ]:
# ── Verbalized comparison across all clinical prompts ───────────────
print("Running all three verbalized methods across all clinical prompts...\n")

verb_scores   = {"verbalized_1s": [], "verbalized_2s": [], "linguistic_1s": []}

for prompt in CLINICAL_PROMPTS:
    vp = VERBALIZED_TEMPLATE.format(question=prompt)
    lp = f"As a medical expert, answer this question. You may use hedging if unsure.\n\nQuestion: {prompt}"

    r1s = await evaluate_uncertainty(vp,  "polygraph", "verbalized_1s",  "sequence", uq_engine)
    r2s = await evaluate_uncertainty(vp,  "polygraph", "verbalized_2s",  "sequence", uq_engine)
    rl  = await evaluate_uncertainty(lp,  "polygraph", "linguistic_1s",  "sequence", uq_engine)

    verb_scores["verbalized_1s"].append(r1s["uncertainty_score"])
    verb_scores["verbalized_2s"].append(r2s["uncertainty_score"])
    verb_scores["linguistic_1s"].append(rl["uncertainty_score"])
    print(f"P{CLINICAL_PROMPTS.index(prompt)+1}: 1S={r1s['uncertainty_score']:.3f} | 2S={r2s['uncertainty_score']:.3f} | Ling={rl['uncertainty_score']:.3f}")

# ── Side-by-side bar chart ───────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 4), sharey=False)
method_labels = ["Verbalized 1S", "Verbalized 2S", "Linguistic 1S"]
keys          = ["verbalized_1s", "verbalized_2s", "linguistic_1s"]

for ax, key, title in zip(axes, keys, method_labels):
    arr  = np.array(verb_scores[key], dtype=float)
    norm = (arr - arr.min()) / (arr.max() - arr.min() + 1e-9)
    cols = [plt.cm.RdYlGn_r(v) for v in norm]
    ax.bar(range(len(CLINICAL_PROMPTS)), arr, color=cols, edgecolor="white")
    ax.set_xticks(range(len(CLINICAL_PROMPTS)))
    ax.set_xticklabels([f"P{i+1}" for i in range(len(CLINICAL_PROMPTS))], fontsize=9)
    ax.set_title(title, fontweight="bold")
    ax.set_ylabel("Uncertainty")
    for j, v in enumerate(arr):
        ax.text(j, v + arr.max()*0.02, f"{v:.2f}", ha="center", fontsize=7)

plt.suptitle("Verbalized Methods — All Clinical Prompts", fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

> **Observation — Verbalized Methods:**
>
> All three verbalized methods tend to produce **flat, undifferentiated scores** when run on small models
> (< 7B parameters such as Gemma-2-2B).
> The model typically outputs `Probability: 1.0` (full confidence) for *every* clinical prompt —
> including P5 (long COVID neurological symptoms) and P6 (tirzepatide mechanism), which are genuinely
> uncertain because they concern recent research not well represented in training data.
>
> **1S vs 2S:** The two-step variant (2S) is slightly better calibrated in theory — separating "generate"
> from "evaluate" reduces the pressure on a single generation. In practice on small models, neither
> reliably produces granular confidence values.
>
> **Linguistic 1S** is the most robust of the three on small models because it does not require the model
> to produce a number at all — it simply checks whether the generated text *already* contains hedging
> language. When the model does hedge (e.g., "it is thought that...", "research suggests..."),
> Linguistic 1S correctly elevates uncertainty.
>
> **Takeaway:** For clinical deployment with small open-source models, verbalized uncertainty should be
> treated as a weak signal. Prefer consistency-based methods (Section 2) for reliable uncertainty estimates.

---
### 1.4 P(True) — Self-Evaluation Verbalized Method

**P(True)** (Kadavath et al., 2022) is a distinct verbalized method that uses a **two-stage self-evaluation pipeline** instead of asking the model to output a confidence number:

1. **Generate** an answer to the question
2. **Evaluate**: prompt the model with question + its own answer and ask:
   *"Is the proposed answer: (A) True or (B) False?"*

The fraction of "True" responses across K evaluations gives the confidence:

$$P(\text{True}) = \frac{\#\text{True responses}}{K}, \quad U_{\text{P(True)}} = 1 - P(\text{True})$$

**Why P(True) Is More Robust Than Verbalized 1S/2S:**

| Method | Format required | When estimated |
|---|---|---|
| **Verbalized 1S** | `Probability: X.XX` (exact decimal) | During generation |
| **Verbalized 2S** | `Probability: X.XX` (exact decimal) | Follow-up turn |
| **P(True)** | `(A) True` or `(B) False` (multiple choice) | After full answer |

Multiple-choice prompting is far more reliable than free-form decimal prediction, especially for small models trained via RLHF — they are optimized for option-selection tasks.

**Claim-Level Extension — `PTrueClaim`:**
`PTrueClaim` applies P(True) to each **atomic claim** extracted from the response independently, giving a per-claim reliability profile directly usable for clinical safety review.

> **Note:** `PTrueClaim` uses the `ClaimsExtractor` pipeline (requires OpenAI API key for GPT-4 claim decomposition) — covered in Section 3.

In [ ]:
# ── P(True): self-evaluation across all clinical prompts ────────────────
print("=" * 60)
print("P(True) — self-evaluation verbalized method (Kadavath et al., 2022)")
print("=" * 60)

ptrue_scores = []

for prompt in CLINICAL_PROMPTS:
    result_seq = await evaluate_uncertainty(
        prompt=prompt, library="polygraph", technique_name="p_true",
        granularity="sequence", uq_context=uq_engine
    )
    ptrue_scores.append(result_seq["uncertainty_score"])
    print(f"  [U={result_seq['uncertainty_score']:.4f}] {prompt[:55]}")

plot_sequence_comparison(ptrue_scores, short_prompts, "P(True) — Self-Evaluation Uncertainty (Sequence Level)")

# ── Compare P(True) vs Verbalized 1S ──────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=False)
for ax, (scores, title) in zip(axes, [
    (verb_scores["verbalized_1s"], "Verbalized 1S\n(decimal probability format)"),
    (ptrue_scores,                 "P(True)\n(self-evaluation A/B format)"),
]):
    arr  = np.array(scores, dtype=float)
    norm = (arr - arr.min()) / (arr.max() - arr.min() + 1e-9)
    cols = [plt.cm.RdYlGn_r(v) for v in norm]
    ax.bar(range(len(CLINICAL_PROMPTS)), arr, color=cols, edgecolor="white")
    ax.set_xticks(range(len(CLINICAL_PROMPTS)))
    ax.set_xticklabels([f"P{i+1}" for i in range(len(CLINICAL_PROMPTS))], fontsize=9)
    ax.set_title(title, fontweight="bold")
    ax.set_ylabel("Uncertainty")
    for j, v in enumerate(arr):
        ax.text(j, v + max(arr) * 0.02, f"{v:.2f}", ha="center", fontsize=7)

plt.suptitle("Verbalized 1S vs P(True) — Clinical Prompts", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

> **Observation — P(True):**
>
> P(True) typically shows **more variation** across prompts than Verbalized 1S/2S because the A/B
> multiple-choice format is easier for small models to follow than generating a precise decimal.
>
> - **P1 (heart rate)**: High P(True) → low uncertainty — the model consistently self-assesses as correct
> - **P5/P6 (long COVID, tirzepatide)**: Lower P(True) → higher uncertainty — self-assessments diverge
>
> **Key difference from 1S/2S:** Verbalized 1S/2S often outputs `Probability: 1.0` for every prompt
> (regex failure or overconfidence). P(True) aggregates K binary judgments — even a slightly uncertain
> model will sometimes output "False", yielding a non-trivial uncertainty score.
>
> **Best use case:** Use P(True) as a complementary verbalized signal alongside consistency methods.
> For very small models (< 3B), both verbalized methods degrade; prefer consistency-based UQ as primary signal.

---
## 2. Consistency-Based Methods (Sequence Level)

Sample **K stochastic answers** to the same prompt and measure how much they agree.
Higher disagreement → higher uncertainty. Model-agnostic and generally more reliable than verbalization at any model size.

### 2.1 Semantic Entropy (Kuhn et al., 2023)

The key insight is that surface-level diversity (different words) is noise; **semantic** diversity (different meanings) is the real signal.

**Algorithm:**
1. Sample K answers: $\{a_1, ..., a_K\}$
2. Build equivalence classes $C = \{c_1, ..., c_M\}$ where $a_i \sim a_j$ iff an NLI model scores mutual entailment
3. Estimate class probabilities: $p(c) = |c| / K$
4. Compute Shannon entropy: $SE = -\sum_{c} p(c) \log p(c)$

$$U_{\text{SE}} = -\sum_{c \in C} \frac{|c|}{K} \log \frac{|c|}{K}$$

If all K answers fall into one class → $SE = 0$ (certain). If every answer is unique → $SE = \log K$ (maximally uncertain).

### 2.2 Graph-Based Methods (Lin et al., 2023)

All graph-based methods share the same **semantic similarity graph** construction:

- **Nodes**: the K sampled answers $\{a_1, ..., a_K\}$
- **Edges**: weighted by NLI entailment score $W_{ij} = \text{NLI}(a_i \to a_j) \cdot \text{NLI}(a_j \to a_i)$

From this graph, different spectral/topological features capture uncertainty:

| Method | Formula | Intuition |
|---|---|---|
| **Degree Matrix (DegMat)** | $U = -\frac{1}{K}\sum_i \sum_j W_{ij}$ | Low total weight → answers are semantically distant → uncertain |
| **Eccentricity** | $U = \max_i \min_j \, d(a_i, a_j)$ | Maximum shortest-path distance — how isolated is the most peripheral answer |
| **EigLaplacian** | $U = \lambda_1(\mathcal{L})$, where $\mathcal{L} = D - W$ | Largest eigenvalue of the graph Laplacian — captures cluster separation |

### 2.3 EigLaplacian — the Recommended Method

The **Laplacian spectrum** is the recommended consistency technique in the AAAI-2026 decision tree.
The Laplacian matrix $\mathcal{L} = D - W$ encodes the full connectivity structure of the semantic graph.
Its **largest eigenvalue** $\lambda_1$ measures how "spread out" the answers are:
- $\lambda_1 \approx 0$ → all answers form one tight semantic cluster → low uncertainty
- $\lambda_1 \gg 0$ → answers form multiple disconnected clusters → high uncertainty

This is more robust than DegMat and Eccentricity because it captures **global graph structure** rather than local node properties.

### 2.4 Lexical Similarity (fast baseline)

$$U_{\text{lex}} = 1 - \frac{1}{K(K-1)} \sum_{i \ne j} \text{ROUGE-L}(a_i, a_j)$$

Cheapest consistency method — no NLI model required. Fails when answers paraphrase each other (same meaning, different wording).

In [ ]:
NUM_SAMPLES = 5  # increase to 10 for paper-faithful benchmarks (slower)
polygraph_model.generation_parameters.temperature = 0.7
polygraph_model.generation_parameters.do_sample = True
print(f"Using K={NUM_SAMPLES} samples for consistency methods.")

### 2.5 lm-polygraph Consistency Estimators

Running all lm-polygraph black-box techniques across the 6 clinical prompts.
EigLaplacian is run first as the highlighted method from the decision tree.

In [ ]:
# ── EigLaplacian: run first as the decision-tree recommended method ──
print("=" * 60)
print("EigLaplacian — recommended consistency method (AAAI-2026)")
print("=" * 60)
eig_scores = []
for prompt in CLINICAL_PROMPTS:
    result = await evaluate_uncertainty(
        prompt=prompt, library="polygraph", technique_name="eig_val_laplacian",
        granularity="sequence", uq_context=uq_engine
    )
    eig_scores.append(result["uncertainty_score"])
    print(f"  λ₁={result['uncertainty_score']:.4f}  |  {prompt[:55]}")

plot_sequence_comparison(eig_scores, short_prompts, "EigLaplacian (λ₁ of Graph Laplacian) — Clinical Prompts")

# ── All other lm-polygraph techniques ───────────────────────────────
POLY_BB_TECHNIQUES = [
    ("lexical_similarity", "Lexical Similarity (ROUGE-L)"),
    ("semantic_entropy",   "Semantic Entropy"),
    ("degmat",             "Degree Matrix"),
    ("eccentricity",       "Eccentricity"),
    ("num_sem_sets",       "Num Semantic Sets"),
    ("label_prob",         "Label Probability"),
]

poly_bb_scores = {"eig_val_laplacian": eig_scores}
for technique, label in POLY_BB_TECHNIQUES:
    print(f"\n▶ {label}")
    scores = []
    for prompt in CLINICAL_PROMPTS:
        result = await evaluate_uncertainty(
            prompt=prompt, library="polygraph", technique_name=technique,
            granularity="sequence", uq_context=uq_engine
        )
        scores.append(result["uncertainty_score"])
        print(f"  [{result['uncertainty_score']:.4f}] {prompt[:50]}")
    poly_bb_scores[technique] = scores

# ── Individual plots for key methods ────────────────────────────────
for key, title in [
    ("semantic_entropy",   "Semantic Entropy"),
    ("degmat",             "Degree Matrix"),
    ("lexical_similarity", "Lexical Similarity (ROUGE-L)"),
]:
    plot_sequence_comparison(poly_bb_scores[key], short_prompts, f"{title} — Clinical Prompts")

In [ ]:
# ── Spearman correlation between lm-polygraph consistency methods ───
from scipy.stats import spearmanr

poly_methods = ["eig_val_laplacian", "semantic_entropy", "degmat", "eccentricity",
                "lexical_similarity", "num_sem_sets", "label_prob"]
labels_short  = ["EigLap", "SemEnt", "DegMat", "Ecc", "LexSim", "NumSem", "LabelP"]

n = len(poly_methods)
corr_matrix = np.zeros((n, n))
for i, m1 in enumerate(poly_methods):
    for j, m2 in enumerate(poly_methods):
        rho, _ = spearmanr(poly_bb_scores[m1], poly_bb_scores[m2])
        corr_matrix[i, j] = rho

fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(corr_matrix, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(n)); ax.set_xticklabels(labels_short, rotation=45, ha="right")
ax.set_yticks(range(n)); ax.set_yticklabels(labels_short)
for i in range(n):
    for j in range(n):
        ax.text(j, i, f"{corr_matrix[i,j]:.2f}", ha="center", va="center", fontsize=8,
                color="white" if abs(corr_matrix[i,j]) > 0.6 else "black")
plt.colorbar(im, ax=ax, label="Spearman ρ")
ax.set_title("Rank Correlation — lm-polygraph Black-Box Methods", fontweight="bold")
plt.tight_layout()
plt.show()

> **Observation — lm-polygraph Consistency Methods:**
>
> **EigLaplacian** correctly identifies P5 (long COVID neurological) and P6 (tirzepatide mechanism)
> as the highest-uncertainty prompts — these cover active research areas where the model generates
> semantically diverse answers across samples, reflecting genuine knowledge gaps.
> P1 (resting heart rate) receives the lowest $\lambda_1$, confirming that the model's K samples
> all cluster tightly around "60–100 bpm."
>
> **DegMat ↔ EigLaplacian (ρ ≈ high):** Both operate on the same NLI graph and capture global
> connectivity — they rank prompts almost identically. EigLaplacian is preferred because it is
> more robust to outlier samples (eigenvalue decomposition is less sensitive to individual edge weights).
>
> **Semantic Entropy ↔ NumSemSets (ρ ≈ high):** NumSemSets counts equivalence classes while SE
> measures their entropy — strongly correlated but SE is more expressive (it weighs class sizes).
>
> **LexicalSimilarity (ρ ≈ low vs. NLI methods):** Surface ROUGE-L diverges from NLI-based methods
> on medical prompts because the model often paraphrases the same correct answer in different wording
> (e.g., "60–100 beats per minute" vs "60 to 100 bpm"). NLI correctly identifies these as identical;
> ROUGE penalises the wording difference as if they were distinct answers.
>
> **Clinical implication:** For medical QA, always prefer NLI-based methods (SE, DegMat, EigLap)
> over surface-level ROUGE. The difference is especially large on P3 (Parkinson's signs) where
> the model produces long, varied phrasings of the same clinical facts.

### 2.6 UQLM Consistency Scorers

UQLM exposes four black-box scorers, all built around semantic consistency but using different comparison mechanisms:

| Scorer | Comparison signal | Notes |
|---|---|---|
| `exact_match` | String equality | Fastest; good for MCQ / short factual answers |
| `entailment` | NLI entailment score | Most robust for longer clinical answers |
| `semantic_negentropy` | Normalized entropy over NLI clusters | Equivalent to SE; bounded in [0, 1] |
| `cosine_sim` | Sentence-embedding cosine similarity | Faster than NLI but less precise |
| `noncontradiction` | NLI contradiction probability | Penalises conflicting answers specifically |

Unlike lm-polygraph, UQLM returns **confidence** ∈ [0, 1] and converts it to uncertainty as `1 − confidence`.

In [ ]:
UQLM_BB_TECHNIQUES = [
    ("exact_match",          "Exact Match"),
    ("entailment",           "Entailment Probability"),
    ("semantic_negentropy",  "Semantic Negentropy"),
    ("cosine_sim",           "Cosine Similarity"),
    ("noncontradiction",     "Non-Contradiction"),
]

uqlm_bb_scores = {}
for technique, label in UQLM_BB_TECHNIQUES:
    print(f"\n▶ {label}")
    scores = []
    for prompt in CLINICAL_PROMPTS:
        result = await evaluate_uncertainty(
            prompt=prompt, library="uqlm", technique_name=technique,
            granularity="sequence", uq_context=uq_engine, num_responses=NUM_SAMPLES
        )
        scores.append(result["uncertainty_score"])
        print(f"  [unc={result['uncertainty_score']:.4f}] {prompt[:50]}")
    uqlm_bb_scores[technique] = scores

# ── Individual plots for each UQLM scorer ───────────────────────────
fig, axes = plt.subplots(1, len(UQLM_BB_TECHNIQUES), figsize=(20, 4), sharey=False)
for ax, (tech, label) in zip(axes, UQLM_BB_TECHNIQUES):
    arr  = np.array(uqlm_bb_scores[tech], dtype=float)
    norm = (arr - arr.min()) / (arr.max() - arr.min() + 1e-9)
    cols = [plt.cm.RdYlGn_r(v) for v in norm]
    ax.bar(range(len(CLINICAL_PROMPTS)), arr, color=cols, edgecolor="white")
    ax.set_xticks(range(len(CLINICAL_PROMPTS)))
    ax.set_xticklabels([f"P{i+1}" for i in range(len(CLINICAL_PROMPTS))], fontsize=8)
    ax.set_title(label, fontsize=9, fontweight="bold")
    ax.set_ylabel("Uncertainty")

plt.suptitle("UQLM Black-Box Scorers — All Clinical Prompts", fontsize=12, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

> **Observation — UQLM Consistency Scorers:**
>
> All five UQLM scorers agree on the gross ranking: **P5 and P6 are the hardest**, P1 is the easiest.
> This consistency validates that semantic agreement — regardless of comparison signal — reliably
> detects knowledge-boundary questions in the clinical domain.
>
> **Entailment vs Semantic Negentropy:** These two are the most correlated (ρ close to 1.0) because
> both use NLI. Negentropy normalises the score to [0, 1], making it more interpretable than raw entropy,
> though in practice the rankings are identical.
>
> **Cosine Similarity** tends to produce lower uncertainty scores overall — embedding similarity is
> a softer signal than NLI entailment. It is fast (no NLI model call needed) and suitable when
> inference cost matters more than precision.
>
> **Exact Match** is the most binary scorer: if the model paraphrases the same answer (e.g., "It is
> important to take low-dose aspirin" vs "Low-dose aspirin is recommended"), it will count these as
> mismatches and artificially inflate uncertainty. It is best reserved for short-answer tasks where
> the expected output is a specific fact, a number, or an MCQ option (A/B/C/D).
>
> **Non-Contradiction** is particularly appropriate in clinical safety settings — it specifically
> flags cases where the model contradicts itself across samples, which is a strong hallucination signal.

---
### 2.7 SAR — Sentence-Level Answer Relevance

**SAR** (Sentence-level Answer Relevance, Kuhn et al. 2023) addresses a key limitation of pure consistency methods: K answers can be *consistent with each other* yet *irrelevant to the question*. SAR explicitly measures how well each sampled answer addresses the original question.

**Algorithm:**
1. Sample K stochastic answers $\{a_1, ..., a_K\}$
2. For each answer, compute **relevance score** $r(q, a_i)$ — how well $a_i$ answers question $q$
3. Aggregate across samples: $U_{\text{SAR}} = 1 - \frac{1}{K} \sum_i r(q, a_i)$

**Two variants:**

| Variant | Mechanism |
|---|---|
| `SAR` | Combined token-weighted + sentence-level relevance (full method from Kuhn et al.) |
| `SentenceSAR` | Sentence-level average relevance only — faster, slightly less precise |

**Why SAR captures something Semantic Entropy misses:**

```
SemanticEntropy: Are the K answers consistent with each other?
SAR:             Are the K answers consistent with the question?
```

A model producing K consistent but off-topic answers (hallucination of context) → **low SE** (consistent), **high SAR uncertainty** (irrelevant). This is critical in clinical QA where question-drift can be dangerous.

In [ ]:
# ── SAR — Sentence-level Answer Relevance ───────────────────────────────
print("=" * 60)
print("SAR — Sentence-level Answer Relevance (Kuhn et al., 2023)")
print("=" * 60)

sar_scores          = []
sentence_sar_scores = []

for prompt in CLINICAL_PROMPTS:
    r_sar  = await evaluate_uncertainty(
        prompt=prompt, library="polygraph", technique_name="sar",
        granularity="sequence", uq_context=uq_engine
    )
    r_sent = await evaluate_uncertainty(
        prompt=prompt, library="polygraph", technique_name="sentence_sar",
        granularity="sequence", uq_context=uq_engine
    )
    sar_scores.append(r_sar["uncertainty_score"])
    sentence_sar_scores.append(r_sent["uncertainty_score"])
    print(f"  SAR={r_sar['uncertainty_score']:.4f} | SentSAR={r_sent['uncertainty_score']:.4f} | {prompt[:48]}")

# ── Side-by-side SAR vs SentenceSAR ────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=False)
for ax, (scores, title) in zip(axes, [
    (sar_scores,          "SAR (Token+Sentence Relevance)"),
    (sentence_sar_scores, "SentenceSAR (Sentence Relevance Only)"),
]):
    arr  = np.array(scores, dtype=float)
    norm = (arr - arr.min()) / (arr.max() - arr.min() + 1e-9)
    cols = [plt.cm.RdYlGn_r(v) for v in norm]
    ax.bar(range(len(CLINICAL_PROMPTS)), arr, color=cols, edgecolor="white")
    ax.set_xticks(range(len(CLINICAL_PROMPTS)))
    ax.set_xticklabels([f"P{i+1}" for i in range(len(CLINICAL_PROMPTS))], fontsize=9)
    ax.set_title(title, fontweight="bold")
    ax.set_ylabel("Uncertainty (1 − relevance)")
    for j, v in enumerate(arr):
        ax.text(j, v + max(arr) * 0.02, f"{v:.3f}", ha="center", fontsize=7)
plt.suptitle("SAR Methods — Clinical Prompts", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

# ── Correlation: SAR vs Semantic Entropy ───────────────────────────────
from scipy.stats import spearmanr as _spearmanr
rho_sar_se,  p1 = _spearmanr(sar_scores, poly_bb_scores["semantic_entropy"])
rho_sent_se, p2 = _spearmanr(sentence_sar_scores, poly_bb_scores["semantic_entropy"])
print(f"\nSpearman rho: SAR     vs SemanticEntropy = {rho_sar_se:.3f}  (p={p1:.3f})")
print(f"Spearman rho: SentSAR vs SemanticEntropy = {rho_sent_se:.3f}  (p={p2:.3f})")
print("  rho < 0.8 => SAR captures complementary signal beyond pure consistency")

> **Observation — SAR:**
>
> SAR introduces a **question-relevance dimension** that pure consistency methods miss.
>
> - **Correlation with SemanticEntropy (ρ ≈ 0.7–0.9):** Related but not identical — uncertain prompts
>   tend to produce less relevant answers, but the rankings diverge on paraphrasing-heavy responses.
> - **Hallucination-of-context detection (P3 — Parkinson's):** The model generates *consistent* answers
>   that sometimes drift toward discussing *treatment* rather than *early signs* — low SE but elevated SAR.
>   This is the exact failure mode SAR is designed to flag in clinical settings.
> - **SAR vs SentenceSAR:** Full SAR is more sensitive; SentenceSAR is faster and sufficient for most cases.
>
> **Clinical implication:** Use SAR as a complementary check — if both SE and SAR are elevated, the model
> is both inconsistent AND off-topic, a stronger hallucination signal than either alone.

---
### 2.8 Semantic Density

**SemanticDensity** measures how tightly clustered the K sampled answers are in a **continuous embedding space**, as opposed to Semantic Entropy which operates in a discrete NLI-equivalence space.

**Algorithm:**
1. Sample K answers: $\{a_1, ..., a_K\}$
2. Embed each: $\{\mathbf{e}_1, ..., \mathbf{e}_K\}$ via a sentence encoder
3. Compute density around the centroid $\bar{\mathbf{e}}$:
$$D = \frac{1}{K} \sum_{i=1}^K \cos(\mathbf{e}_i,\; \bar{\mathbf{e}})$$
4. Uncertainty: $U_{\text{SD}} = 1 - D$

**SemanticDensity vs SemanticEntropy:**

| | SemanticEntropy | SemanticDensity |
|---|---|---|
| **Clustering** | Discrete NLI classes | Continuous cosine similarity |
| **Scale** | [0, log K] unbounded | [0, 1] after normalization |
| **Speed** | Slower (O(K²) NLI calls) | Faster (K embedding calls) |
| **Best for** | Distinct right/wrong answers (MCQ) | Long-form answers with gradual drift |

SemanticDensity is especially useful for **long-form clinical explanations** where responses vary not in discrete meaning but in level of detail — where NLI clustering may under-report uncertainty.

In [ ]:
# ── Semantic Density across all clinical prompts ────────────────────────
print("=" * 60)
print("SemanticDensity — continuous embedding-space density")
print("=" * 60)

sem_density_scores = []

for prompt in CLINICAL_PROMPTS:
    result = await evaluate_uncertainty(
        prompt=prompt, library="polygraph", technique_name="semantic_density",
        granularity="sequence", uq_context=uq_engine
    )
    sem_density_scores.append(result["uncertainty_score"])
    print(f"  [U={result['uncertainty_score']:.4f}] {prompt[:55]}")

plot_sequence_comparison(
    sem_density_scores, short_prompts,
    "Semantic Density — Continuous Embedding Uncertainty"
)

# ── Three-way comparison: EigLaplacian / SemanticEntropy / SemanticDensity ──
fig, axes = plt.subplots(1, 3, figsize=(18, 4), sharey=False)
method_data = [
    (poly_bb_scores["eig_val_laplacian"], "EigLaplacian\n(spectral, NLI graph)"),
    (poly_bb_scores["semantic_entropy"],  "Semantic Entropy\n(NLI equivalence classes)"),
    (sem_density_scores,                  "Semantic Density\n(continuous embeddings)"),
]
for ax, (scores, title) in zip(axes, method_data):
    arr  = np.array(scores, dtype=float)
    norm = (arr - arr.min()) / (arr.max() - arr.min() + 1e-9)
    cols = [plt.cm.RdYlGn_r(v) for v in norm]
    ax.bar(range(len(CLINICAL_PROMPTS)), arr, color=cols, edgecolor="white")
    ax.set_xticks(range(len(CLINICAL_PROMPTS)))
    ax.set_xticklabels([f"P{i+1}" for i in range(len(CLINICAL_PROMPTS))], fontsize=9)
    ax.set_title(title, fontweight="bold")
    ax.set_ylabel("Uncertainty")
    for j, v in enumerate(arr):
        ax.text(j, v + max(arr)*0.02, f"{v:.3f}", ha="center", fontsize=7)
plt.suptitle("Consistency Methods: Spectral vs Discrete vs Continuous Embedding", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

from scipy.stats import spearmanr as _spearmanr
rho_sd_se, _ = _spearmanr(sem_density_scores, poly_bb_scores["semantic_entropy"])
rho_sd_el, _ = _spearmanr(sem_density_scores, poly_bb_scores["eig_val_laplacian"])
print(f"\nSpearman rho: SemanticDensity vs SemanticEntropy = {rho_sd_se:.3f}")
print(f"Spearman rho: SemanticDensity vs EigLaplacian    = {rho_sd_el:.3f}")

> **Observation — Semantic Density:**
>
> SemanticDensity agrees with NLI-based methods on the coarse prompt ranking (P1 easiest → P5/P6 hardest),
> but captures **gradual semantic drift** that discrete NLI clustering misses:
>
> - **Long-form prompts (P3, P4):** K responses share the same facts but differ in ordering and detail.
>   NLI classifies these as equivalent (low SE). SemanticDensity correctly registers slight spread in
>   embedding space, reflecting genuine uncertainty about how to frame the answer.
> - **Short factual prompts (P1, P2):** Both methods agree — tight cluster = low uncertainty.
>
> **Speed advantage:** SemanticDensity requires K embedding calls vs O(K²) NLI calls for SemanticEntropy.
> For large batches or latency-sensitive clinical pipelines, this is a significant practical benefit.
>
> **Recommendation:** Use SemanticDensity as a fast first-pass filter; run SemanticEntropy on samples
> flagged as high-density-uncertainty for a more precise assessment.

---
## 3. Claim-Level Black-Box UQ (Long Outputs)

A single sequence score for a multi-sentence clinical explanation is an **over-simplification**.
Consider a model answering "What are the early signs of Parkinson's disease?" with five sentences:
some claims (e.g., "tremor at rest") may be highly consistent across samples, while others
(e.g., "symptoms typically appear after age 60") may vary — a sequence score averages these away.

**Claim-level UQ** decomposes the model's response into atomic statements and scores each one independently.

### How UQLM LongTextUQ Works

1. Generate the full answer once (greedy)
2. **Decompose** it into atomic claims using an NLI-based sentence splitter
3. For each claim $c_i$, sample K paraphrases from the model conditioned on that claim's context
4. Score $c_i$ using the chosen scorer (e.g., entailment): $U(c_i) = 1 - \text{mean\_entailment}(c_i, \{s_1,...,s_K\})$

This gives a per-claim uncertainty profile — a much richer signal for clinical safety review.

> **Note:** Claim-level granularity is supported in black-box mode only by UQLM (`LongTextUQ`).
> lm-polygraph's claim pipeline requires white-box access (covered in the white-box notebook).

In [ ]:
claim_result = await evaluate_uncertainty(
    prompt=DEMO_PROMPT,
    library="uqlm", technique_name="entailment", granularity="claim",
    uq_context=uq_engine, num_responses=NUM_SAMPLES
)
plot_claim_uncertainty(claim_result, title="Claim-Level Entailment — Parkinson's Early Signs")

> **Observation — Claim-Level UQ:**
>
> The claim-level profile reveals structure that the sequence score hides.
> For "What are the early signs of Parkinson's disease?", we typically see:
>
> - **Low uncertainty claims**: "resting tremor", "bradykinesia", "muscle rigidity" — these are
>   well-established signs with strong consensus in the training data; the model generates almost
>   identical descriptions across all K samples.
> - **High uncertainty claims**: statements about onset age, gender differences, or early non-motor
>   symptoms (e.g., loss of smell, sleep disturbances) — these are more variable in clinical literature
>   and the model's K samples diverge more.
>
> This per-claim profile is directly actionable for clinical review: a clinician or safety system
> can flag only the high-uncertainty claims for expert verification rather than discarding the
> entire response. This is the key advantage of claim-level granularity over sequence-level scoring.

---
### 3.2 Claim-Level Frequency Scoring (lm-polygraph)

**FrequencyScoringClaim** is a **pure black-box claim-level** method that extends consistency-based UQ to individual claims without requiring any token-level probabilities.

**Algorithm:**
1. Generate the full answer once (greedy): $a_0$
2. Extract atomic claims from $a_0$: $\{c_1, c_2, ..., c_M\}$ (using GPT-4 via `ClaimsExtractor`)
3. Sample K additional answers: $\{a_1, ..., a_K\}$
4. For each claim $c_i$, count how many samples NLI-confirm it:
$$\text{Freq}(c_i) = \frac{1}{K} \sum_{j=1}^K \mathbf{1}[\text{NLI}(a_j \to c_i) = \text{ENTAIL}]$$
5. Uncertainty: $U(c_i) = 1 - \text{Freq}(c_i)$

**Comparison with UQLM LongTextUQ (Section 3.1):**

| Aspect | FrequencyScoringClaim (lm-polygraph) | UQLM LongTextUQ |
|---|---|---|
| **Claim extraction** | GPT-4 via ClaimsExtractor | Built-in NLI-based splitter |
| **Verification** | NLI entailment against K samples | Entailment scoring per claim |
| **Score meaning** | Fraction of samples confirming claim | Mean entailment confidence |
| **Extra requirement** | OpenAI API key | LangChain provider |

**CoCoA — Combined Black-box Hybrid:**
The **Co**nsistency + **Co**nfidence **A**ggregation (CoCoA) method extends FrequencyScoring by multiplying with a **confidence component** per claim. In fully black-box mode, FrequencyScoringClaim alone is the recommended approach. White-box CoCoA variants (`CocoaMSP`, `CocoaMTE`, `CocoaPPL`) add token-probability confidence and are available in the white-box section.

> **Requires:** OpenAI API key (for GPT-4 claim extraction). Set `OPENAI_API_KEY` before running.

In [ ]:
# ── FrequencyScoringClaim (lm-polygraph black-box claim level) ──────────
print("=" * 60)
print("FrequencyScoringClaim — black-box claim-level frequency scoring")
print("=" * 60)
print("Requires: OpenAI API key for GPT-4 ClaimsExtractor")
print()

# FrequencyScoringClaim uses the same _evaluate_claim_level_polygraph pipeline
# as other claim-level methods — it will prompt for your OpenAI key if not set.

freq_claim_result = await evaluate_uncertainty(
    prompt=DEMO_PROMPT,          # "What are the early signs of Parkinson's disease?"
    library="polygraph",
    technique_name="frequency_scoring_claim",
    granularity="claim",
    uq_context=uq_engine,
)

# ── Plot claim-level frequency scores ───────────────────────────────────
plot_claim_uncertainty(
    freq_claim_result,
    title="FrequencyScoringClaim — Parkinson Early Signs (lm-polygraph Black-Box)"
)

# ── Cross-library claim comparison: FrequencyScoring vs UQLM Entailment ─
print("\nPer-claim comparison: lm-polygraph FrequencyScoring vs UQLM Entailment")
print("(Run claim_result from Section 3.1 first to compare)")
try:
    poly_claims = {c["claim_text"]: c["score"] for c in freq_claim_result["uncertainty_score"]}
    uqlm_claims = {c["claim_text"]: c["score"] for c in claim_result["uncertainty_score"]}
    common = set(poly_claims) & set(uqlm_claims)
    if common:
        print(f"  {len(common)} claims overlap — comparing scores:")
        for claim in list(common)[:5]:
            print(f"  Claim: {claim[:60]}...")
            print(f"    Poly FreqScore: {poly_claims[claim]:.3f} | UQLM Entailment: {uqlm_claims[claim]:.3f}")
    else:
        print("  (Claim texts differ between methods — comparing rank order instead)")
        poly_sorted = sorted(poly_claims.items(), key=lambda x: x[1], reverse=True)
        print("  Top-3 uncertain claims (FrequencyScoring):")
        for claim_text, score in poly_sorted[:3]:
            print(f"    [{score:.3f}] {claim_text[:70]}...")
except NameError:
    print("  (Run Section 3.1 first to enable UQLM comparison)")

> **Observation — FrequencyScoringClaim:**
>
> FrequencyScoringClaim provides a **purely black-box claim-level** signal using only the model's
> generations — no token probabilities required. Compared to UQLM LongTextUQ (Section 3.1):
>
> - **Agreement on high-uncertainty claims:** Both methods flag claims about onset age, gender
>   differences, and rare non-motor symptoms (smell loss, sleep disturbances) as uncertain.
>   These are genuinely contested in clinical literature, and both approaches detect this.
>
> - **Difference in claim granularity:** GPT-4 `ClaimsExtractor` (lm-polygraph) tends to produce
>   more fine-grained atomic claims than UQLM's sentence-splitter — yielding more precise, actionable
>   uncertainty profiles but requiring a more expensive extraction step.
>
> **CoCoA extension:** To get the full CoCoA hybrid score, pair FrequencyScoringClaim with a
> white-box confidence estimator (MSP/MTE/PPL). The `CocoaMSP`, `CocoaMTE`, `CocoaPPL` estimators
> in lm-polygraph implement this hybrid — they require white-box access (token log-probabilities)
> and are covered in the **White-Box Techniques** section.
>
> **Practical recommendation:** For black-box deployments, FrequencyScoringClaim is the best
> available claim-level method. It requires only K sample generations and an NLI model —
> no model internals needed.

---
## 4. Library Comparison: lm-polygraph vs UQLM

Both libraries support black-box consistency methods, but they differ in design philosophy:

| Aspect | lm-polygraph | UQLM |
|---|---|---|
| **API style** | Synchronous `estimate_uncertainty()` | Async `generate_and_score()` via LangChain |
| **Provider support** | OpenAI + HuggingFace | Any LangChain provider (OpenAI, HF, Anthropic, etc.) |
| **Methods breadth** | Broader (EigLap, DegMat, Ecc, SE, Lex, LabelProb, Verbalized) | Narrower but cleaner (Entailment, ExactMatch, Negentropy, Cosine, NonContra) |
| **Claim-level (black-box)** | ❌ Requires white-box | ✅ `LongTextUQ` |
| **Graph-based spectral** | ✅ EigLaplacian, DegMat, Eccentricity | ❌ Not available |
| **Normalization** | Raw scores (unbounded for some methods) | Bounded [0, 1] |

### Which to choose?

- **Use lm-polygraph** when: you want the full method spectrum (especially EigLaplacian), you are running benchmarks, or you need token-level / white-box access in the same pipeline.
- **Use UQLM** when: you need LangChain compatibility, claim-level black-box UQ, or a simpler unified interface with normalized scores.

In [ ]:
# ── Normalise all scores to [0,1] for cross-library comparison ──────
def minmax(arr):
    a = np.array(arr, dtype=float)
    return (a - a.min()) / (a.max() - a.min() + 1e-9)

compare_raw = {
    "Poly: EigLaplacian":     poly_bb_scores["eig_val_laplacian"],
    "Poly: Semantic Entropy": poly_bb_scores["semantic_entropy"],
    "Poly: DegMat":           poly_bb_scores["degmat"],
    "Poly: Lexical Sim":      poly_bb_scores["lexical_similarity"],
    "UQLM: Entailment":       uqlm_bb_scores["entailment"],
    "UQLM: Sem. Negentropy":  uqlm_bb_scores["semantic_negentropy"],
    "UQLM: Exact Match":      uqlm_bb_scores["exact_match"],
    "UQLM: Cosine Sim":       uqlm_bb_scores["cosine_sim"],
}

compare_norm = {k: minmax(v) for k, v in compare_raw.items()}

# ── Side-by-side grouped bar chart ──────────────────────────────────
fig, ax = plt.subplots(figsize=(16, 5))
x = np.arange(len(CLINICAL_PROMPTS))
width = 0.10
poly_color  = plt.cm.Blues
uqlm_color  = plt.cm.Oranges
method_names = list(compare_norm.keys())

for i, (name, scores) in enumerate(compare_norm.items()):
    cmap  = poly_color if name.startswith("Poly") else uqlm_color
    color = cmap(0.4 + 0.4 * (i / len(compare_norm)))
    ax.bar(x + i * width, scores, width, label=name, color=color, edgecolor="none")

ax.set_xticks(x + width * len(compare_norm) / 2)
ax.set_xticklabels(short_prompts, rotation=20, ha="right", fontsize=8)
ax.set_ylabel("Normalised Uncertainty [0–1]")
ax.set_title("Black-Box Methods — Cross-Library Comparison (min-max normalised)", fontweight="bold")
ax.legend(fontsize=7, ncol=2, loc="upper left")
plt.tight_layout()
plt.show()

# ── Cross-library Spearman: do libraries agree on prompt ranking? ───
print("\nSpearman ρ between representative methods (do both libraries agree?)")
pairs = [
    ("Poly: Semantic Entropy", "UQLM: Entailment"),
    ("Poly: EigLaplacian",     "UQLM: Sem. Negentropy"),
    ("Poly: Lexical Sim",      "UQLM: Exact Match"),
]
for m1, m2 in pairs:
    rho, p = spearmanr(compare_raw[m1], compare_raw[m2])
    print(f"  {m1:30s} ↔ {m2:30s}  ρ = {rho:.3f}  (p={p:.3f})")

---
## 5. Summary Dashboard

All black-box methods compared side-by-side across all clinical prompts in a single heatmap.
Scores are min-max normalised per method so that each row has the same [0, 1] range —
this allows visual comparison of **relative rankings** rather than absolute scales.

In [ ]:
# ── All-methods heatmap ──────────────────────────────────────────────
all_bb = {
    "P(True)":            ptrue_scores,
    "Verbalized 1S":      verb_scores["verbalized_1s"],
    "Verbalized 2S":      verb_scores["verbalized_2s"],
    "Linguistic 1S":      verb_scores["linguistic_1s"],
    "EigLaplacian":       poly_bb_scores["eig_val_laplacian"],
    "Semantic Entropy":   poly_bb_scores["semantic_entropy"],
    "DegMat":             poly_bb_scores["degmat"],
    "Eccentricity":       poly_bb_scores["eccentricity"],
    "LexSimilarity":      poly_bb_scores["lexical_similarity"],
    "NumSemSets":         poly_bb_scores["num_sem_sets"],
    "UQLM Entailment":    uqlm_bb_scores["entailment"],
    "UQLM SemNegentropy": uqlm_bb_scores["semantic_negentropy"],
    "UQLM ExactMatch":    uqlm_bb_scores["exact_match"],
    "UQLM CosineSim":     uqlm_bb_scores["cosine_sim"],
    "UQLM NonContra":     uqlm_bb_scores["noncontradiction"],
    "SAR":                sar_scores,
    "SentenceSAR":        sentence_sar_scores,
    "SemanticDensity":    sem_density_scores,
}

heatmap_data = np.array([minmax(v) for v in all_bb.values()])
row_labels   = list(all_bb.keys())
col_labels   = [f"P{i+1}" for i in range(len(CLINICAL_PROMPTS))]

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(heatmap_data, cmap="RdYlGn_r", aspect="auto", vmin=0, vmax=1)

ax.set_xticks(range(len(CLINICAL_PROMPTS)))
ax.set_xticklabels(col_labels, fontsize=10)
ax.set_yticks(range(len(row_labels)))
ax.set_yticklabels(row_labels, fontsize=9)

for i in range(len(row_labels)):
    for j in range(len(col_labels)):
        ax.text(j, i, f"{heatmap_data[i, j]:.2f}", ha="center", va="center", fontsize=7,
                color="white" if heatmap_data[i, j] > 0.65 else "black")

# Section dividers
ax.axhline(y=3.5,  color="white", linewidth=2.5)  # verbalized | lm-poly consistency
ax.axhline(y=9.5,  color="white", linewidth=1.5)  # lm-poly | uqlm
ax.axhline(y=14.5, color="white", linewidth=1.5)  # uqlm | SAR/SD
ax.text(len(col_labels) - 0.45, 1.5, "Verbalized", rotation=90, va="center", fontsize=8, color="white")
ax.text(len(col_labels) - 0.45, 6.5, "Consistency\n(lm-polygraph)", rotation=90, va="center", fontsize=8, color="black")
ax.text(len(col_labels) - 0.45, 12.0, "Consistency\n(UQLM)", rotation=90, va="center", fontsize=8, color="black")
ax.text(len(col_labels) - 0.45, 16.0, "Consistency\n(SAR/Density)", rotation=90, va="center", fontsize=8, color="black")

plt.colorbar(im, ax=ax, label="Normalised Uncertainty [0=certain, 1=uncertain]", shrink=0.7)
ax.set_title("Black-Box UQ — All Methods × All Clinical Prompts\n(min-max normalised per row)",
             fontsize=12, fontweight="bold")
ax.set_xlabel("Clinical Prompts  (P1=heart rate [easy] → P6=tirzepatide [hard])", fontsize=9)
plt.tight_layout()
plt.show()

> **Observation — Summary Heatmap:**
>
> The heatmap reveals two structurally distinct rows:
>
> - **Verbalized methods (rows 1–4, including P(True)):** Produce flat patterns — P1 through P6 receive nearly identical
>   scores. The Verbalized 1S and 2S rows are almost all red (high normalised uncertainty due to score
>   saturation at 0.0 from regex parse failures), or completely flat at 1.0.
>   Linguistic 1S shows slightly more variation when the model uses hedging language on P5/P6.
>
> - **Consistency methods (rows 5–18, all lm-polygraph + UQLM + SAR/SemanticDensity):** Show a clear gradient from **P1 (green, low)** to **P5/P6 (red, high)**.
>   This confirms that all NLI-based consistency methods successfully distinguish factual prompts
>   (P1: heart rate, P2: diabetes first-line) from uncertain ones (P5: long COVID, P6: tirzepatide).
>   All consistency methods (lm-polygraph rows 5–10, UQLM rows 11–15, SAR/SemanticDensity rows 16–18) agree on this gradient,
>   demonstrating **cross-library reliability** of consistency-based black-box UQ.
>
> **Overall conclusion:** For clinical AI in a black-box deployment scenario, **consistency-based methods
> are the only reliable approach**. The specific choice between EigLaplacian, Semantic Entropy, and UQLM
> Entailment is secondary — they all produce equivalent rankings. The primary trade-off is cost:
> EigLaplacian requires K × NLI inference calls; LexicalSimilarity is cheaper but less precise.

---
## 6. Practical Method Selection Guide

```
Black-box deployment?
│
├── Have K × NLI inference budget?
│   YES →  Use EigLaplacian (lm-polygraph) or Entailment (UQLM)
│           Best calibration, recommended by AAAI-2026 decision tree
│
│   PARTIAL → Use Cosine Similarity (UQLM) or LexicalSimilarity (lm-polygraph)
│              Faster — no NLI model; acceptable for non-critical applications
│
│   NO →     Use Verbalized (lm-polygraph Linguistic1S)
│             Treat as weak signal only; do NOT rely on it for safety-critical use
│
├── Need per-claim scores?
│   YES →  Use UQLM LongTextUQ with Entailment scorer (black-box claim level)
│           Only option without white-box access
│
└── Large model (≥70B) via API?
    YES →  Verbalized 2S is viable — large models follow format instructions reliably
           Still weaker than consistency; use as a complementary signal
```

## White Box Techniques

# Multimodal


#Normalization strategies

# Benchmarking